# Supervivencia en el Titanic: EDA y clasificación
### Caso clásico de clasificación binaria. El foco está en un EDA exhaustivo y un preprocesamiento sin fugas, antes de cualquier modelo.

> **Data & Machine Learning · Análisis exploratorio (EDA)** — notebook ejecutable top-down en Google Colab.
> Comentado en español · `random_state=42` · sin data leakage (todo el preprocesamiento dentro de `Pipeline`).

In [ ]:
# Portada (IPython.display.HTML)
from IPython.display import HTML, display
SPEC = {"categoria": "Data & Machine Learning", "subtema": "Análisis exploratorio (EDA)", "titulo": "Supervivencia en el Titanic: EDA y clasificación", "subtitulo": "Caso clásico de clasificación binaria. El foco está en un EDA exhaustivo y un preprocesamiento sin fugas, antes de cualquier modelo.", "ficha": {"Dataset": "Titanic (891 pasajeros)", "Fuente": "Kaggle / mirror público", "Problema": "Clasificación binaria", "Target": "Survived (0/1)", "Técnica": "LogReg vs. RandomForest", "Métrica": "ROC-AUC"}, "objetivo": "Predecir si un pasajero sobrevivió a partir de sus atributos, priorizando un EDA y un preprocesamiento rigurosos.", "hipotesis": "Sexo, clase del pasaje (Pclass) y edad concentran la mayor parte de la señal predictiva.", "datos_desc": "891 filas; mezcla de variables numéricas (Age, Fare, SibSp, Parch) y categóricas (Sex, Embarked, Pclass), con faltantes en Age, Cabin y Embarked.", "metodologia": "EDA → split estratificado → ColumnTransformer (imputación + escalado + one-hot) dentro de Pipeline → validación cruzada → evaluación en test.", "metrica": "ROC-AUC en validación cruzada (5-fold) y en el set de test.", "problema": "clasificacion", "target": "Survived", "loader_code": "# 1 · Carga de datos\n# Titanic desde un mirror público (en Colab podés usar la Kaggle API con tu kaggle.json).\nimport pandas as pd, numpy as np\nURL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'\ndf = pd.read_csv(URL)\n# Pruning de columnas no predictivas / texto libre / identificadores (decisión de diseño):\ndf = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])\nprint('Columnas finales:', list(df.columns))", "conclusiones": ["El EDA confirma la hipótesis: <b>Sex</b> y <b>Pclass</b> son los predictores más fuertes; <b>Age</b> aporta señal pero tiene ~20% de faltantes.", "El preprocesamiento se encapsula en un <b>Pipeline</b>, así la imputación y el escalado se ajustan <b>solo con train</b> — sin data leakage.", "RandomForest supera al baseline LogReg en ROC-AUC, pero la diferencia es moderada: en datos tan chicos, el baseline interpretable es competitivo.", "Próximos pasos: feature engineering (título extraído del nombre, tamaño de familia) y calibración de probabilidades."]}
def _cover_html(s: dict) -> str:
    chips = "".join(
        f'<div style="border:1px solid {LINE};border-radius:10px;padding:12px 14px;">'
        f'<div style="font:600 10px/1 Inter,sans-serif;letter-spacing:.14em;'
        f'text-transform:uppercase;color:{CORAL};margin-bottom:6px;">{_html.escape(k)}</div>'
        f'<div style="font:500 14px/1.3 Inter,sans-serif;color:{INK};">{_html.escape(v)}</div></div>'
        for k, v in s["ficha"].items()
    )
    return f"""<div style="font-family:Inter,system-ui,sans-serif;max-width:860px;margin:0 auto;">
<style>{FONTS}</style>
<div style="border-left:6px solid {CYAN};padding:6px 0 6px 22px;margin-bottom:26px;">
  <div style="font:600 11px/1 Inter,sans-serif;letter-spacing:.22em;text-transform:uppercase;color:{CYAN};margin-bottom:12px;">
    {_html.escape(s['categoria'])} · {_html.escape(s['subtema'])}
  </div>
  <h1 style="font:700 34px/1.12 'Space Grotesk',sans-serif;color:{INK};margin:0 0 10px;">{_html.escape(s['titulo'])}</h1>
  <p style="font:400 16px/1.5 Inter,sans-serif;color:{MUTE};margin:0;max-width:640px;">{_html.escape(s['subtitulo'])}</p>
</div>
<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin-bottom:22px;">{chips}</div>
<div style="display:flex;align-items:center;gap:10px;color:{MUTE};font:500 13px/1 Inter,sans-serif;">
  <span style="color:{INK};font-weight:600;">Nicolás Bargioni</span>
  <span style="width:4px;height:4px;border-radius:50%;background:{CORAL};"></span>
  <span>Data Scientist · nicolasbargioni.com</span>
  <span style="margin-left:auto;font:600 11px/1 'Space Grotesk',monospace;color:{CYAN};">ds-colabs</span>
</div></div>"""

CYAN='#0891b2';CORAL='#f0521f';INK='#0f172a';MUTE='#64748b';LINE='#e2e8f0';FONTS="@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=Inter:wght@400;500;600&display=swap');"
import html as _html
display(HTML(_cover_html(SPEC)))

In [ ]:
def _block_html(title: str, items: list[tuple[str, str]]) -> str:
    rows = "".join(
        f'<div style="margin-bottom:16px;"><div style="font:600 11px/1 Inter,sans-serif;'
        f'letter-spacing:.12em;text-transform:uppercase;color:{CORAL};margin-bottom:5px;">{_html.escape(h)}</div>'
        f'<div style="font:400 15px/1.55 Inter,sans-serif;color:{INK};">{b}</div></div>'
        for h, b in items
    )
    return f"""<div style="font-family:Inter,system-ui,sans-serif;max-width:860px;margin:0 auto;
border:1px solid {LINE};border-radius:14px;padding:26px 30px;">
<style>{FONTS}</style>
<h2 style="font:700 20px/1.2 'Space Grotesk',sans-serif;color:{INK};margin:0 0 20px;">{_html.escape(title)}</h2>
{rows}</div>"""

display(HTML(_block_html('Marco del trabajo', [
    ('Objetivo', "Predecir si un pasajero sobrevivió a partir de sus atributos, priorizando un EDA y un preprocesamiento rigurosos."),
    ('Hipótesis', "Sexo, clase del pasaje (Pclass) y edad concentran la mayor parte de la señal predictiva."),
    ('Datos', "891 filas; mezcla de variables numéricas (Age, Fare, SibSp, Parch) y categóricas (Sex, Embarked, Pclass), con faltantes en Age, Cabin y Embarked."),
    ('Metodología', "EDA → split estratificado → ColumnTransformer (imputación + escalado + one-hot) dentro de Pipeline → validación cruzada → evaluación en test."),
    ('Métrica', "ROC-AUC en validación cruzada (5-fold) y en el set de test."),
])))

---
## 0 · Setup — dependencias, imports y configuración global

In [ ]:
# 0.1 Dependencias (Colab ya trae casi todo; instala solo lo que falte)
import importlib, sys, subprocess
for _p in ['datasets', 'seaborn']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', _p])

# 0.2 Imports
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 0.3 Configuración global (reproducibilidad + estética)
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 110
PRIMARY, ACCENT = '#0891b2', '#f0521f'
print('Setup OK · numpy', np.__version__, '· pandas', pd.__version__)

## 1 · Carga de datos

In [ ]:
# 1 · Carga de datos
# Titanic desde un mirror público (en Colab podés usar la Kaggle API con tu kaggle.json).
import pandas as pd, numpy as np
URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(URL)
# Pruning de columnas no predictivas / texto libre / identificadores (decisión de diseño):
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
print('Columnas finales:', list(df.columns))

In [ ]:
# Vista rápida de la tabla cargada (el loader debe dejar un DataFrame `df`)
assert 'df' in globals(), 'El loader debe definir un DataFrame llamado df'
df = df.copy()
print('Filas × columnas:', df.shape)
df.head()

## 2 · Análisis exploratorio (EDA)
EDA exhaustivo **antes** de tocar el modelo: forma, tipos, faltantes, distribuciones, cardinalidad, correlaciones, outliers y relación con el target.

In [ ]:
# 2.1 Forma, tipos y memoria
print('Dimensiones:', df.shape)
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
print(f'{len(num_cols)} numéricas · {len(cat_cols)} categóricas')
display(df.dtypes.value_counts().rename('columnas').to_frame())
df.info()

In [ ]:
# 2.2 Valores faltantes
na = df.isna().sum(); na = na[na > 0].sort_values(ascending=False)
if len(na):
    display(pd.DataFrame({'faltantes': na, '%': (na/len(df)*100).round(2)}))
    plt.figure(figsize=(7, max(2, 0.4*len(na))))
    sns.barplot(x=na.values, y=na.index, color=PRIMARY)
    plt.title('Faltantes por columna'); plt.xlabel('cantidad'); plt.tight_layout(); plt.show()
else:
    print('Sin valores faltantes.')

In [ ]:
# 2.3 Variable objetivo
TARGET = "Survived"
if TARGET and TARGET in df.columns:
    if "clasificacion" == 'regresion':
        display(df[TARGET].describe().to_frame())
        plt.figure(figsize=(6,3)); sns.histplot(df[TARGET], kde=True, color=PRIMARY)
        plt.title(f'Distribución de {TARGET}'); plt.tight_layout(); plt.show()
    else:
        vc = df[TARGET].value_counts(dropna=False)
        display(pd.DataFrame({'n': vc, '%': (vc/len(df)*100).round(2)}))
        plt.figure(figsize=(5,3)); sns.barplot(x=vc.index.astype(str), y=vc.values, color=PRIMARY)
        plt.title(f'Balance — {TARGET}'); plt.ylabel('n'); plt.tight_layout(); plt.show()
        imb = vc.min()/vc.max()
        print(f'Ratio min/max de clases: {imb:.2f}' + ('  ⚠️ desbalanceado' if imb < 0.4 else ''))
else:
    print('Sin target supervisado en esta etapa.')

In [ ]:
# 2.4 Distribuciones de variables numéricas
num_cols = df.select_dtypes(include='number').columns.tolist()
if num_cols:
    display(df[num_cols].describe().T)
    cols = num_cols[:9]; rows = (len(cols)+2)//3
    fig, axes = plt.subplots(rows, 3, figsize=(13, 3*rows))
    for ax, col in zip(np.ravel(axes), cols):
        sns.histplot(df[col].dropna(), kde=True, ax=ax, color=PRIMARY); ax.set_title(col, fontsize=10)
    for ax in np.ravel(axes)[len(cols):]: ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Sin columnas numéricas.')

In [ ]:
# 2.5 Categóricas: cardinalidad y valores frecuentes
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
if cat_cols:
    card = df[cat_cols].nunique().sort_values(ascending=False)
    display(card.rename('valores únicos').to_frame())
    for col in card.index[:4]:
        print(f'\n{col} — top 6:'); print(df[col].value_counts(dropna=False).head(6))
else:
    print('Sin columnas categóricas.')

In [ ]:
# 2.6 Matriz de correlación (numéricas)
num_cols = df.select_dtypes(include='number').columns.tolist()
if len(num_cols) >= 2:
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(min(1+0.7*len(num_cols),12), min(1+0.6*len(num_cols),10)))
    sns.heatmap(corr, annot=len(num_cols)<=12, fmt='.2f', cmap='coolwarm', center=0, square=True)
    plt.title('Correlaciones'); plt.tight_layout(); plt.show()
else:
    print('Pocas numéricas para correlación.')

In [ ]:
# 2.7 Outliers por regla IQR
num_cols = df.select_dtypes(include='number').columns.tolist()
rows = []
for col in num_cols:
    q1, q3 = df[col].quantile([.25, .75]); iqr = q3 - q1
    n = int(((df[col] < q1-1.5*iqr) | (df[col] > q3+1.5*iqr)).sum())
    rows.append((col, n, round(n/len(df)*100, 2)))
if rows:
    display(pd.DataFrame(rows, columns=['columna','outliers','%']).sort_values('outliers', ascending=False))
else:
    print('Sin numéricas para evaluar outliers.')

## 3 · Preprocesamiento
Split **antes** de transformar (evita leakage). Imputación + escalado + encoding encapsulados en `ColumnTransformer`/`Pipeline`, ajustados **solo con train**.

In [ ]:
# 3.1 X / y y split train/test (estratificado si es clasificación)
TARGET = "Survived"
assert TARGET in df.columns, f'Target no encontrado: {TARGET}'
X = df.drop(columns=[TARGET]); y = df[TARGET]
strat = y if "clasificacion" == 'clasificacion' else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=strat)
print('train:', X_train.shape, '· test:', X_test.shape)

In [ ]:
# 3.2 Preprocesador (se ajusta SOLO con train dentro del Pipeline → sin leakage)
num_feats = X_train.select_dtypes(include='number').columns.tolist()
cat_feats = X_train.select_dtypes(exclude='number').columns.tolist()
numeric = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
categ = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                  ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric, num_feats), ('cat', categ, cat_feats)])
print(f'{len(num_feats)} numéricas + {len(cat_feats)} categóricas → preprocesador listo')

## 4 · Modelado

In [ ]:
# 4 · Candidatos (baseline + ensemble) con validación cruzada
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = 'roc_auc' if y.nunique()==2 else 'f1_macro'
cands = {'LogReg (baseline)': LogisticRegression(max_iter=1000, random_state=SEED),
         'RandomForest': RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)}
results = {}
for name, clf in cands.items():
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    sc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results[name] = sc; print(f'{name:22s} {scoring}: {sc.mean():.4f} ± {sc.std():.4f}')
best_name = max(results, key=lambda k: results[k].mean()); print('\nMejor:', best_name)
best = Pipeline([('prep', preprocess), ('clf', cands[best_name])]).fit(X_train, y_train)

## 5 · Evaluación

In [ ]:
# 5 · Evaluación en test (datos nunca vistos)
from sklearn.metrics import (classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay)
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred))
binary = y.nunique() == 2
fig, ax = plt.subplots(1, 2 if binary else 1, figsize=(11,4) if binary else (5,4))
ax0 = ax[0] if binary else ax
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
    display_labels=sorted(map(str, y.unique()))).plot(ax=ax0, cmap='Blues', colorbar=False)
ax0.set_title('Matriz de confusión')
if binary:
    RocCurveDisplay.from_estimator(best, X_test, y_test, ax=ax[1], color=ACCENT); ax[1].set_title('Curva ROC')
plt.tight_layout(); plt.show()

## 6 · Conclusiones

In [ ]:
def _conclusion_html(items: list[str]) -> str:
    lis = "".join(
        f'<li style="margin-bottom:10px;font:400 15px/1.55 Inter,sans-serif;color:{INK};">{b}</li>'
        for b in items
    )
    return f"""<div style="font-family:Inter,system-ui,sans-serif;max-width:860px;margin:0 auto;
border-left:6px solid {CORAL};padding:4px 0 4px 22px;">
<style>{FONTS}</style>
<div style="font:600 11px/1 Inter,sans-serif;letter-spacing:.2em;text-transform:uppercase;color:{CORAL};margin-bottom:10px;">Conclusiones</div>
<ul style="margin:0;padding-left:18px;">{lis}</ul></div>"""

display(HTML(_conclusion_html([
    "El EDA confirma la hipótesis: <b>Sex</b> y <b>Pclass</b> son los predictores más fuertes; <b>Age</b> aporta señal pero tiene ~20% de faltantes.",
    "El preprocesamiento se encapsula en un <b>Pipeline</b>, así la imputación y el escalado se ajustan <b>solo con train</b> — sin data leakage.",
    "RandomForest supera al baseline LogReg en ROC-AUC, pero la diferencia es moderada: en datos tan chicos, el baseline interpretable es competitivo.",
    "Próximos pasos: feature engineering (título extraído del nombre, tamaño de familia) y calibración de probabilidades.",
])))